# FinLLM — Financial News Sentiment Analysis with LoRA
### Assignment 2: Individual Development of a Large Language Model (LLM)
**Student Name:** Sameet Pisal  
**PRN:** 202401120018  
**Roll Number:** 41  
**Institute:** MIT Academy of Engineering  
**Submission Date:** 21 November 2026  

---
## 1. Project Overview
This notebook provides full GPU-accelerated training for **FinLLM**, an open-source Large Language Model specialized for 3-class financial news sentiment analysis (Positive, Negative, Neutral).  
It adapts **Qwen/Qwen2.5-0.5B-Instruct** using **LoRA (Low-Rank Adaptation)** on the official **Financial PhraseBank** dataset.

## 2. Environment Setup & GPU Verification

In [ ]:
!nvidia-smi
!pip install -q torch transformers datasets peft accelerate scikit-learn pandas matplotlib

## 3. Dataset Download & Stratification (Financial PhraseBank)

In [ ]:
import os, zipfile, urllib.request, pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

# Download official Financial PhraseBank v1.0
url = 'https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip'
os.makedirs('data/raw', exist_ok=True)
zip_path = 'data/raw/FinancialPhraseBank-v1.0.zip'

if not os.path.exists(zip_path):
    print('Downloading Financial PhraseBank...')
    urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('data/raw')

txt_path = 'data/raw/FinancialPhraseBank-v1.0/Sentences_50Agree.txt'
rows = []
with open(txt_path, 'r', encoding='iso-8859-1') as f:
    for line in f:
        if '@' in line:
            s, l = line.rsplit('@', 1)
            rows.append({'sentence': s.strip(), 'label': l.strip().lower()})

df = pd.DataFrame(rows).drop_duplicates(subset=['sentence'])
print(f'Total clean sentences: {len(df)}')
print(df['label'].value_counts())

# Stratified 80/10/10 split (seed=42)
train_val, test_df = train_test_split(df, test_size=180, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(train_val, test_size=80, stratify=train_val['label'], random_state=42)

print(f'Train: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}')

## 4. Model Loading & LoRA Configuration

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto'
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Supervised Fine-Tuning with Answer-Only Loss Masking

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

SYSTEM_PROMPT = (
    'You are a financial sentiment analysis model. '
    'Classify the provided financial text based on its implications for the '
    "company's financial performance. "
    'Respond with exactly one label: positive, negative or neutral.'
)

class FinancialDataset(Dataset):
    def __init__(self, data_df, tokenizer, max_len=128):
        self.items = []
        for _, r in data_df.iterrows():
            msgs = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': r['sentence']},
                {'role': 'assistant', 'content': r['label']}
            ]
            text = tokenizer.apply_chat_template(msgs, tokenize=False)
            enc = tokenizer(text, truncation=True, max_length=max_len, add_special_tokens=False)
            ids = enc['input_ids']
            
            prefix = tokenizer.encode('<|im_start|>assistant\n', add_special_tokens=False)
            labels = [-100] * len(ids)
            for i in range(len(ids) - len(prefix) + 1):
                if ids[i:i+len(prefix)] == prefix:
                    for j in range(i+len(prefix), len(ids)):
                        labels[j] = ids[j]
                    break
            self.items.append({'input_ids': torch.tensor(ids), 'labels': torch.tensor(labels)})
            
    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

def pad_collate(batch):
    ml = max(x['input_ids'].size(0) for x in batch)
    ids, lbls, mask = [], [], []
    for x in batch:
        l = x['input_ids'].size(0)
        p = ml - l
        ids.append(torch.cat([torch.full((p,), tokenizer.pad_token_id), x['input_ids']]))
        lbls.append(torch.cat([torch.full((p,), -100), x['labels']]))
        mask.append(torch.cat([torch.zeros(p), torch.ones(l)]))
    return {'input_ids': torch.stack(ids), 'labels': torch.stack(lbls), 'attention_mask': torch.stack(mask)}

train_loader = DataLoader(FinancialDataset(train_df.iloc[:300], tokenizer), batch_size=4, shuffle=True, collate_fn=pad_collate)
optimizer = AdamW(model.parameters(), lr=2e-4)

model.train()
print('Starting GPU fine-tuning...')
for epoch in range(2):
    total_loss = 0
    for step, b in enumerate(train_loader):
        b = {k: v.to(device) for k, v in b.items()}
        optimizer.zero_grad()
        loss = model(**b).loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if (step+1) % 10 == 0:
            print(f'Epoch {epoch+1} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}')
    print(f'Epoch {epoch+1} Average Loss: {total_loss/len(train_loader):.4f}')

## 6. Save Adapter & Export for Local FinLLM Project

In [ ]:
ADAPTER_OUT = 'finllm_adapter_colab'
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
!zip -r finllm_adapter_colab.zip finllm_adapter_colab
print('Adapter saved and zipped successfully!')
print('To transfer back to your local machine: download finllm_adapter_colab.zip and unzip into FinLLM/models/finllm_adapter/')